# Limpieza de textos y tokenizacion


> Empleamos diferentes metodos para la limpieza de los documentos, corrigiendo así signos de puntuación, emojis o cualquier otro elemento que interfeira el análisis del mismo.




## Exportamos librerías y definimos función de limpieza del texto

In [ ]:
import string
import pandas as pd
import re
import html
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import nltk
import pickle

# Unicode Emoticones
EMOJI_PATTERN = re.compile(
    "["
    u"\U0001F600-\U0001F64F"   # emoticons
    u"\U0001F300-\U0001F5FF"   # símbolos y pictogramas
    u"\U0001F680-\U0001F9FF"   # transport, misc symbols
    u"\U00002600-\U000027BF"   # símbolos varios
    "]+",
    flags=re.UNICODE
)


def get_wordnet_pos(treebank_tag): # Devuelve el significado del diccionario de wordnet
    """Convierte el POS tag de NLTK al formato de WordNet."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN



def clean_docs(df: pd.DataFrame) -> list:
  lemmatizer = WordNetLemmatizer()

  # Eliminamos caracteres especiales. urls, htmls, emojis, etc.
  cleaned_docs = []
  for text in df["texto"]:
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = EMOJI_PATTERN.sub(' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    cleaned_docs.append(text)

  # Conversión de documentos a minusculas
  raw_docs = [cleaned_docs[x].lower() for x in range(len(cleaned_docs))]

  # Eliminar numeros
  raw_docs = [re.sub(r'\d+', ' ', doc) for doc in raw_docs]
  raw_docs = [re.sub(r'\s+', ' ', doc).strip() for doc in raw_docs]

  # Tokenización
  tokenize_docs = [word_tokenize(doc) for doc in raw_docs]

  # Remover puntuación
  regex = re.compile('[%s]' % re.escape(string.punctuation))
  tokenize_docs_no_punctuation = []
  for review in tokenize_docs:
    new_review = []
    for token in review:
      new_token = regex.sub(u'', token)
      if not new_token == u'':
        new_review.append(new_token)
    tokenize_docs_no_punctuation.append(new_review)

  # Remover stopwords
  stop_words = set(stopwords.words('english'))
  keep = {"no", "not", "nor", "never", "neither", "very", "too", "but", "however", "although"}
  stop_words -= keep
  tokenized_docs_no_stopwords = []
  for doc in tokenize_docs_no_punctuation:
    new_term_vector = []
    for word in doc:
      if not word in stop_words:
        new_term_vector.append(word)

    tokenized_docs_no_stopwords.append(new_term_vector)

  # Lemmatización
  tokenized_docs_clean = []
  for doc in tokenized_docs_no_stopwords:
    pos_tags = pos_tag(doc)  # [('running', 'VBG'), ('good', 'JJ'), ...]
    lemmatized = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    tokenized_docs_clean.append(lemmatized)

  return tokenized_docs_clean


## Limpieza de textos

In [ ]:
# Carga de dataset
df = pd.read_csv("datasets/dataset_unificado_corregido.csv")

# División de registros. Separando 10000 registros clases balanceadas y con etiqueta de entrenamiento
pos_train = df[(df["sentimiento"]=="pos") & (df["split"]=="train")].sample(n=5000, random_state=42)
neg_train = df[(df["sentimiento"]=="neg") & (df["split"]=="train")].sample(n=5000, random_state=42)
df_balanced = pd.concat([pos_train, neg_train]).sample(frac=1, random_state=42).reset_index(drop=True)
tokenized_docs_train_clean = clean_docs(df_balanced)


# Separar datos de test
pos_test = df[(df["sentimiento"]=="pos") & (df["split"]=="test")].sample(n=2500, random_state=42)
neg_test = df[(df["sentimiento"]=="neg") & (df["split"]=="test")].sample(n=2500, random_state=42)
df_test = pd.concat([pos_test, neg_test]).sample(frac=1, random_state=42).reset_index(drop=True)
tokenized_docs_test_clean = clean_docs(df_test)


# Creación de y_test y y_train
df_balanced["tag"] = df_balanced["sentimiento"].map({"pos":1, "neg":0})
y_train  = [int(df_balanced["tag"][x]) for x in range(len(df_balanced["texto"]))]

df_test["tag"] = df_test["sentimiento"].map({"pos":1, "neg":0})
y_test = [int(df_test["tag"][x]) for x in range(len(df_test["texto"]))]


## Guardado de limpieza de reviews

In [ ]:
# Guardado de documentos tokenizados y listas de y_train e y_test
with open('data/processed/tokenized_docs_train_clean.pkl', 'wb') as f:
    pickle.dump(tokenized_docs_train_clean, f)
print("Variable tokenized_docs_train_clean guardada correctamente")

with open('data/processed/tokenized_docs_test_clean.pkl', 'wb') as f:
    pickle.dump(tokenized_docs_test_clean, f)
    print("Variable tokenized_docs_test_clean guardada correctamente")

with open('data/processed/y_train.pkl', 'wb') as f:
    pickle.dump(y_train, f)
    print("Variable y_train guardada correctamente")

with open('data/processed/y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)
    print("Variable y_test guardada correctamente")


Más adelante se pueden cargar estas variables para un uso posterior:
```python
import pickle
# Ejemplo
with open('tokenized_docs_clean.pkl', 'rb') as f:
    loaded_tokenized_docs_clean = pickle.load(f)

print("Variable 'loaded_tokenized_docs_clean' cargada.")
```

# Doc2Vec-Vectorizacion de los datos


> Uso de Doc2Vec para vectorizar documentos. En este apartado tokenizamos una parte que se usará para el entrenamiento y otra para la validación



## Entrenamiento Modelo Doc2Vec (Vectorización de documentos)

In [ ]:
!pip install gensim
import pickle
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import numpy as np

with open('data/processed/tokenized_docs_test_clean.pkl', 'rb') as f:
    tokenized_docs_test_clean = pickle.load(f)

with open('data/processed/tokenized_docs_train_clean.pkl', 'rb') as f:
    tokenized_docs_train_clean = pickle.load(f)

# Entrenamiento y vectorización de documentos usando Doc2Vec
labeled_data = [TaggedDocument(words=tokenized_docs_train_clean[x], tags=[x]) for x in range(len(tokenized_docs_train_clean))]
model = Doc2Vec(vector_size=384, window=5, min_count=2, workers=1, seed=42, epochs=150, dm=1) # Ajustes de entrenamiento de Doc2Vec
model.build_vocab(labeled_data)
model.train(labeled_data, total_examples=model.corpus_count, epochs=model.epochs)
print("Entrenamiento Doc2Vec finalizado correctamente")

## Guardado de variables


In [ ]:
# Guardar modelo para no repetir este proceso
model.save("proyect-package/dsr/doc2vec_model")
print("Modelo Doc2Vec guardado en proyect-package/dsr/doc2vec_model")

# Vectorización usando el modelo ya entrenado
# infer_vector se aplica a cada documento individualmente
X_test = np.array([model.infer_vector(doc) for doc in tokenized_docs_test_clean])
X_train = np.array([model.dv[i] for i in range(len(tokenized_docs_train_clean))])

print("Variable X_test creada correctamente")

with open('data/processed/X_train.pkl', 'wb') as f:
    pickle.dump(X_train, f)
print("X_train guaradado correctamente")
with open('data/processed/X_test.pkl', 'wb') as f:
    pickle.dump(X_test, f)
print("X_test guardado correctamente")

# **Modelo de Algoritmo Knn**

> Uso de libreria sklear, modelo de clasificacion KNeighbors


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import pickle
with open('data/processed/X_train.pkl', 'rb') as f:
    X_train = pickle.load(f)

with open('data/processed/y_train.pkl', 'rb') as f:
    y_train = pickle.load(f)

with open('data/processed/X_test.pkl', 'rb') as f:
    X_test = pickle.load(f)

with open('data/processed/y_test.pkl', 'rb') as f:
    y_test = pickle.load(f)

KNeig = KNeighborsClassifier(n_neighbors=71, leaf_size=30, weights="distance",  algorithm="brute", p=1, metric="cosine")

KNeig.fit(X_train, y_train) # Entrenamiento

# Guardar modelo KNN en el paquete dsr
with open('proyect-package/dsr/knn_model.pkl', 'wb') as f:
    pickle.dump(KNeig, f)
print("Modelo KNN guardado en proyect-package/dsr/knn_model.pkl")

## Métricas del modelo


> Comprobamos el rendimiento del modelo



In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt

# Predecir
y_pred = KNeig.predict(X_test)

# Get probabilities for ROC curve and AUC score
y_proba = KNeig.predict_proba(X_test)[:, 1] # Probability of the positive class

# Métricas en una sola línea
print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))

# Accuracy por separado si lo necesitas
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# AUC-ROC Score
auc_roc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC Score: {auc_roc:.4f}")

# Matriz de confusión visual
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['negativo', 'positivo'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión')
plt.show()

# Curva ROC visual
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_roc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Modelo de Regresion Logistica

In [ ]:
from sklearn.linear_model import LogisticRegression
import pickle

with open('data/processed/X_train.pkl', 'rb') as f:
    X_train = pickle.load(f)

with open('data/processed/y_train.pkl', 'rb') as f:
    y_train = pickle.load(f)

with open('data/processed/X_test.pkl', 'rb') as f:
    X_test = pickle.load(f)

with open('data/processed/y_test.pkl', 'rb') as f:
    y_test = pickle.load(f)

logreg = LogisticRegression(penalty="l2", C=0.01, max_iter=1000, solver="saga")

logreg.fit(X_train, y_train) # Entrenamiento

# Guardar modelo LR en el paquete dsr
with open('proyect-package/dsr/lr_model.pkl', 'wb') as f:
    pickle.dump(logreg, f)
print("Modelo LR guardado en proyect-package/dsr/lr_model.pkl")

## Métricas del modelo

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt

# Predecir
y_pred = logreg.predict(X_test)

# Get probabilities for ROC curve and AUC score
y_proba = logreg.predict_proba(X_test)[:, 1] # Probability of the positive class

# Métricas en una sola línea
print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))

# Accuracy por separado si lo necesitas
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# AUC-ROC Score
auc_roc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC Score: {auc_roc:.4f}")

# Matriz de confusión visual
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['negativo', 'positivo'])
disp.plot(cmap='Oranges')
plt.title('Matriz de Confusión')
plt.show()

# Curva ROC visual
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_roc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


# Red neuronal (No aplicado a proyecto final)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

input_dim = len(X_train[0])  # p.ej. 300

modelo = Sequential([
    Dense(256, activation='relu', input_shape=(input_dim,)),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

modelo.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

X_val = np.array(X_test)
y_val = np.array(y_test)

historial = modelo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Bosquejo de Interfaz

In [ ]:
class Drs:
  #Modelo Unico de Doc2Vec
  __doc2_model = Doc2Vec.load("doc2vec_model")
  __knn_model = pickle.load(open("knn_model.sav", "rb"))

  def __knn(self, vector):
    return KNeig.predict(vector)

  def __vectorizacion(self, text): # Tokenizacion y vertorizacion de texto (Input)
    doc = text.split()
    return model.infer_vector(doc)

  def predic(self, text):
    vector = self.__vectorizacion(text)
    return self.__knn_model.predict(vector)

